## SwiGLU 8 results

This notebook reports the local-fitting block check (Part A) and the slice-only versus fitted model runs at eight removal targets (Part B). It reads completed results only.

#### Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'src' / 'mlp_replacement').is_dir()
)
S8_RESULTS = PROJECT_ROOT / 'data/results/workflows/model/swiglu-8'
BLOCK_CHECK_PATH = S8_RESULTS / 'block-check-001/result.json'
PREPARED_PATH = S8_RESULTS / 'prepare-001/result.json'
# Earlier evidence for the "why" section, read only; a missing file falls back to the documented numbers.
SWIGLU3_PATH = PROJECT_ROOT / 'data/results/workflows/model/swiglu-3/run-001.json'
OPERATOR_DISTILLATION_PATH = PROJECT_ROOT / 'data/results/notebook-block-study/operator-distillation-v4.json'
ARMS = ('fitted', 'slice')
TARGETS = (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8)
RUN_PATHS = {
    (arm, target): S8_RESULTS / f'{arm}-target-{target}-run-001/result.json'
    for arm in ARMS for target in TARGETS
}
ORIGINAL_WIDTH = 8192
# Decided interpretation (decision log, 2026-10-10 and 2026-10-11): each target is judged by
# the 8K KL gap at 100M; the monitoring-KL gap is stable if it moves less than one percentage
# point between 50M and 100M. The 2% threshold is a judgment call.
GAP_THRESHOLD = 0.02
STABILITY_THRESHOLD = 0.01
# Pre-run resource plan in docs/experiments/model/swiglu/swiglu-8.md (expected GPU-hours).
PLANNED_GPU_HOURS = {'block-check': (16.0, 31.0), 'prepare': (3.0, 8.0), 'train': (20.0, 24.0)}
ARM_LABELS = {'fitted': 'Fitted', 'slice': 'Slice only'}
ARM_COLORS = dict(zip(ARMS, sns.color_palette('colorblind', len(ARMS))))

In [ ]:
def acceptance_frame(groups):
    rows = []
    for group, summary in groups.items():
        for check in summary['checks']:
            rows.append({
                'Group': group,
                'Check': check['check'],
                'Metric': check.get('metric'),
                'Layer': check.get('layer'),
                'Width': check.get('width'),
                'Tokens': check.get('tokens'),
                'Value': check['value'],
                'Reference': check.get('reference'),
                'Relative difference': check.get('relative_difference'),
                'Tolerance': check.get('tolerance'),
                'Passed': check['passed'],
            })
    return pd.DataFrame(rows, columns=[
        'Group', 'Check', 'Metric', 'Layer', 'Width', 'Tokens', 'Value', 'Reference',
        'Relative difference', 'Tolerance', 'Passed',
    ])


def acceptance_summary(frame):
    return (
        frame.groupby('Group', sort=False)
        .agg(**{
            'Checks': ('Passed', 'size'),
            'Passed': ('Passed', 'sum'),
            'Largest relative difference': ('Relative difference', 'max'),
        })
        .reset_index()
    )


def number(value):
    return f'{value:.6g}' if isinstance(value, float) else value


def workflow_hours(artifact):
    return (
        pd.Timestamp(artifact['updated_at_utc']) - pd.Timestamp(artifact['created_at_utc'])
    ).total_seconds() / 3600


ACCEPTANCE_FORMAT = {
    'Value': number, 'Reference': number,
    'Relative difference': '{:.2e}', 'Tolerance': '{:.0e}',
}

#### Part A: block check

##### Block-check artifact

In [ ]:
if not BLOCK_CHECK_PATH.is_file():
    raise FileNotFoundError(f'Missing SwiGLU-8 block check: {BLOCK_CHECK_PATH}')

block_check = json.loads(BLOCK_CHECK_PATH.read_text(encoding='utf-8'))
if (block_check.get('workflow'), block_check.get('command')) != ('swiglu-8', 'block-check'):
    raise ValueError('Unexpected SwiGLU-8 block-check artifact')

block_results = block_check['results']
long_context = block_results['long_context_data']
setting_grid = block_check['configuration']['block_check']['settings']
PART_A_SEEDS = block_check['configuration']['block_check']['seeds']
# SwiGLU-7's recipe; every reduction below is relative to its seed mean.
CURRENT_SETTING = 'c1e-3-128'
expected_fits = 6 * sum(1 if setting['method'] == 'least_squares' else len(PART_A_SEEDS)
                        for setting in setting_grid.values())


def describe(name):
    setting = setting_grid[name]
    if setting['method'] == 'least_squares':
        return {'Method': 'least squares', 'Context': setting['calibration_context'], 'Schedule': '-',
                'Learning rate': None, 'Batch': None}
    return {'Method': 'gradient', 'Context': setting['calibration_context'], 'Schedule': setting['scheduler'],
            'Learning rate': setting['learning_rate'], 'Batch': setting['batch_size']}


print(f"Block check: {BLOCK_CHECK_PATH.relative_to(PROJECT_ROOT)}")
print(f"Status: {block_check['status']}" + (f"; error: {block_check['error']}" if block_check.get('error') else ''))
print(f"Fingerprint: {block_check['run_fingerprint']}")
print(f"Unfitted slices: {len(block_results.get('slices', []))}; fits: {len(block_results.get('fits', []))} of {expected_fits}")
print(f"8K selection stream sha256 (Part A scoring only): {long_context['selection_stream']['sha256']}")
print(f"8K KL stream sha256 (final evaluation; compare with Quantization-1): {long_context['kl_stream']['sha256']}")

##### Acceptance and data audit

In [ ]:
block_acceptance = acceptance_frame(block_results.get('acceptance', {}))
display(
    acceptance_summary(block_acceptance).style.hide(axis='index')
    .format({'Largest relative difference': '{:.2e}'}, na_rep='-')
    .set_caption('Part A acceptance: A1-A3 reproduce SwiGLU-7; P5 is the teacher scored against itself')
)
display(block_acceptance.style.hide(axis='index').format(ACCEPTANCE_FORMAT, na_rep='-'))

audit = long_context['audit']
stream_roles = {
    'kl_stream': 'final 8K KL (Part B, Quantization-1)',
    'calibration_stream': '8K training pairs',
    'validation_stream': '8K early stopping',
    'selection_stream': 'Part A 8K score',
}
stream_df = pd.DataFrame([
    {
        'Stream': name.removesuffix('_stream'),
        'Role': role,
        'Tokens': long_context[name]['token_count'],
        'Documents': long_context[name]['documents_consumed'],
        'First record': long_context[name].get('first_record', audit['kl_record_range'][0]),
        'Last record': long_context[name].get('last_record', audit['kl_record_range'][1]),
        'Skipped 128-token records': long_context[name].get('excluded_records_skipped', 0),
        'sha256': long_context[name]['sha256'][:16],
    }
    for name, role in stream_roles.items() if name in long_context
])
display(stream_df.style.hide(axis='index').set_caption('C4 shard-0 8K streams in document order'))
print(f"No two 8K streams share a document: {audit.get('document_disjoint')}")
print('128-token windows drawn from 8K-KL documents (pre-existing SwiGLU-7 overlap):',
      audit['local_windows_from_kl_records'])

##### Recipe comparison

In [ ]:
fits_df = pd.DataFrame([
    {
        'Setting': row['setting'],
        'Seed': row['seed'],
        'Layer': row['layer'],
        'Width ratio': row['width_ratio'],
        'KL @ 128': row['kl_128'],
        'KL @ 8K': row['kl_8192'],
        'NMSE @ 128': row['validation_128']['nmse'],
        'NMSE @ 8K': row['validation_8192']['nmse'],
        'Best epoch': row.get('best_epoch'),
        'Epochs': row.get('epochs_completed'),
        'Updates': row.get('updates'),
        'Fit seconds': row.get('fit_seconds'),
    }
    for row in [*block_results.get('slices', []), *block_results.get('fits', [])]
])
fits_df['Setting'] = pd.Categorical(fits_df['Setting'], ['slice', *setting_grid], ordered=True)
setting_df = (
    fits_df.groupby('Setting', observed=True)
    .agg(**{
        'Fits': ('Layer', 'size'),
        'Mean KL @ 8K': ('KL @ 8K', 'mean'),
        'Mean KL @ 128': ('KL @ 128', 'mean'),
        'Mean NMSE @ 128': ('NMSE @ 128', 'mean'),
        'Mean NMSE @ 8K': ('NMSE @ 8K', 'mean'),
        'Mean best epoch': ('Best epoch', 'mean'),
        'Mean fit seconds': ('Fit seconds', 'mean'),
    })
    .reset_index()
)
display(
    setting_df.style.hide(axis='index')
    .format({
        'Mean KL @ 8K': '{:.6f}', 'Mean KL @ 128': '{:.6f}', 'Mean NMSE @ 128': '{:.4f}',
        'Mean NMSE @ 8K': '{:.4f}', 'Mean best epoch': '{:.1f}', 'Mean fit seconds': '{:.0f}',
    }, na_rep='-')
    .set_caption('Averages over the six layer-width combinations and all seeds; "slice" is the unfitted teacher subset')
)
combination_kl = fits_df.pivot_table(index='Setting', columns=['Layer', 'Width ratio'], values='KL @ 8K',
                                     aggfunc='mean', observed=True)
display(combination_kl.style.format('{:.6f}').set_caption('Seed-mean single-block KL @ 8K (selection stream) by combination'))

##### Adoption

In [ ]:
adoption = block_results['adoption']
stage_two = block_results['stage_two_indicator']
adoption_df = pd.DataFrame([
    {
        'Setting': name,
        **describe(name),
        'Mean reduction @ 8K': row['mean_reduction']['8192'],
        'Mean seed noise @ 8K': row['mean_seed_noise'],
        'Mean reduction @ 128': row['mean_reduction']['128'],
        'Worst reduction': row['worst_reduction'],
        'Qualifies': row['qualifies'],
        'Reasons': '; '.join(row['reasons']) or '-',
    }
    for name, row in adoption['settings'].items()
])
display(
    adoption_df.style.hide(axis='index')
    .format({
        'Mean reduction @ 8K': '{:+.2%}', 'Mean seed noise @ 8K': '{:.2%}', 'Mean reduction @ 128': '{:+.2%}',
        'Worst reduction': '{:+.2%}', 'Learning rate': '{:g}', 'Batch': '{:.0f}',
    }, na_rep='-')
    .set_caption(
        'Decided rule (decision log, 2026-10-11): seed-mean 8K reduction above 5% and above the seed noise; '
        'no combination more than 5% worse at 128 or 8K'
    )
)
print(f"Reference: {adoption['reference_setting']} (SwiGLU-7's recipe), seed means over seeds {PART_A_SEEDS}")
print(f"Qualifying settings: {', '.join(adoption['qualifying']) or 'none'}")
print(f"Selected Part B setting: {adoption['selected_recipe']}")
print(
    f"Stage-2 indicator for {stage_two['setting']}: applicable={stage_two['applicable']}, "
    f"overfitting remains={stage_two['overfitting_remains']}, layers={stage_two.get('overfitting_layers', [])}"
)

In [ ]:
reduction_df = pd.DataFrame([
    {
        'Setting': name,
        'Context': entry['context'],
        'Layer': entry['layer'],
        'Width': entry['width'],
        'Reduction (%)': 100 * entry['reduction'],
    }
    for name, row in adoption['settings'].items() for entry in row['reductions']
])
reduction_df['Combination'] = [
    f'L{layer} @ {width / ORIGINAL_WIDTH:.2f}' for layer, width in zip(reduction_df['Layer'], reduction_df['Width'])
]
combination_order = [
    f'L{layer} @ {width / ORIGINAL_WIDTH:.2f}'
    for layer, width in sorted(set(zip(reduction_df['Layer'], reduction_df['Width'])))
]
figure, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
for axis, context in zip(axes, (8192, 128)):
    matrix = (
        reduction_df[reduction_df['Context'] == context]
        .pivot(index='Setting', columns='Combination', values='Reduction (%)')
        .loc[list(adoption['settings']), combination_order]
    )
    sns.heatmap(
        matrix, ax=axis, cmap='RdBu', center=0, annot=True, fmt='.1f',
        cbar=axis is axes[-1], cbar_kws={'label': 'KL reduction vs current setting (%)'},
    )
    axis.set(title=f'Single-block KL at {context:,} tokens', xlabel='Layer @ width ratio', ylabel='')
figure.suptitle(f'Positive values improve on the seed mean of {CURRENT_SETTING}', y=1.01)
figure.tight_layout()
plt.show()

##### Fitting curves

In [ ]:
curve_rows = []
for row in block_results['fits']:
    if row['seed'] != 21 or row.get('calibration_context') != 128 or not row.get('history'):
        continue
    for entry in row['history']:
        if entry['phase'] == 'full':
            curve_rows.append({
                'Setting': row['setting'],
                'Layer': row['layer'],
                'Width ratio': row['width_ratio'],
                'Epoch': entry['epoch'],
                'Validation NMSE': entry['validation_nmse'],
            })
curve_df = pd.DataFrame(curve_rows)
curve_settings = [name for name in setting_grid if name in set(curve_df['Setting'])]
curve_colors = dict(zip(curve_settings, sns.color_palette('colorblind', len(curve_settings))))
curve_layers = sorted(curve_df['Layer'].unique())
curve_widths = sorted(curve_df['Width ratio'].unique())
figure, axes = plt.subplots(len(curve_widths), len(curve_layers), figsize=(15, 7.5), sharex=True, squeeze=False)
for row_index, width in enumerate(curve_widths):
    for column_index, layer in enumerate(curve_layers):
        axis = axes[row_index, column_index]
        for name in curve_settings:
            rows = curve_df[
                (curve_df['Setting'] == name) & (curve_df['Layer'] == layer) & (curve_df['Width ratio'] == width)
            ]
            axis.plot(rows['Epoch'], rows['Validation NMSE'], color=curve_colors[name], linewidth=1.5, label=name)
        least_squares = fits_df[
            (fits_df['Setting'] == 'ols-128') & (fits_df['Layer'] == layer) & (fits_df['Width ratio'] == width)
        ]
        if len(least_squares):
            axis.axhline(least_squares['NMSE @ 128'].iloc[0], color='black', linestyle='--', linewidth=1,
                         label='ols-128')
        axis.set(
            title=f'Layer {layer}, width {width:.2f}',
            xlabel='Epoch' if row_index == len(curve_widths) - 1 else '',
            ylabel='Validation NMSE @ 128' if column_index == 0 else '',
        )
axes[0, 0].legend(frameon=False, fontsize=8, ncol=2)
figure.suptitle('Held-out local error during fitting: seed 21, 128-token settings (cosine runs a fixed 32 epochs)', y=1.01)
figure.tight_layout()
plt.show()

#### Why these local-fitting settings

This section explains each local-fitting setting used for the SwiGLU replacements: what was tested, on which data, and what the evidence shows. Settings are judged by single-block KL, the teacher KL when only one block is replaced. Every Part A choice is made on selection data that the final evaluation never uses (decision log, 2026-10-11).

| Data | Used for | In the final evaluation? |
| --- | --- | --- |
| 128-token calibration pairs (393,216) | Training pairs at 128 tokens | No |
| 128-token operator validation (6,144 pairs) | Early stopping at 128 tokens | No |
| 128-token allocation selection (3,072 tokens) | Single-block KL at 128 tokens; width allocation | No |
| 8K calibration stream (48 x 8,192 tokens) | Training pairs at 8K | No |
| 8K validation stream (1 x 8,192 tokens) | Early stopping at 8K | No |
| 8K selection stream (24 x 8,192 tokens) | Single-block KL at 8K, the deciding score | No |
| 8K KL stream (24 x 8,192 tokens, Quantization-1) | Final 8K KL at 100M | Yes |

Seeds change only the order of training batches; the data, the kept neurons, and the starting weights are fixed.

In [ ]:
earlier_evidence = {}
for name, path in (('swiglu-3', SWIGLU3_PATH), ('operator-distillation', OPERATOR_DISTILLATION_PATH)):
    if path.is_file():
        earlier_evidence[name] = json.loads(path.read_text(encoding='utf-8'))
    else:
        print(f'Earlier evidence not available locally: {path.relative_to(PROJECT_ROOT)}; documented numbers are cited.')
print(f"Earlier evidence loaded: {', '.join(earlier_evidence) or 'none'}")

##### Learning rate, schedule, and training context

In [ ]:
reference_noise = np.mean([
    row['seed_sd'] / row['mean'] for row in adoption['reference_seed_summaries'] if row['context'] == 8192
])
grid_rows = [{
    'Setting': CURRENT_SETTING, 'Training context': 128, 'Schedule': 'constant', 'Learning rate': 1e-3,
    'Reduction @ 8K (%)': 0.0, 'Reduction @ 128 (%)': 0.0, 'Seed noise @ 8K (%)': 100 * reference_noise,
}]
for name, row in adoption['settings'].items():
    setting = setting_grid[name]
    if setting['method'] != 'gradient' or setting['batch_size'] != setting_grid[CURRENT_SETTING]['batch_size']:
        continue
    grid_rows.append({
        'Setting': name,
        'Training context': setting['calibration_context'],
        'Schedule': setting['scheduler'],
        'Learning rate': setting['learning_rate'],
        'Reduction @ 8K (%)': 100 * row['mean_reduction']['8192'],
        'Reduction @ 128 (%)': 100 * row['mean_reduction']['128'],
        'Seed noise @ 8K (%)': 100 * row['mean_seed_noise'],
    })
grid_df = pd.DataFrame(grid_rows)
schedule_colors = dict(zip(('constant', 'cosine'), sns.color_palette('colorblind', 2)))
figure, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for axis, context in zip(axes, (128, 8192)):
    for schedule, rows in grid_df[grid_df['Training context'] == context].groupby('Schedule'):
        rows = rows.sort_values('Learning rate')
        axis.errorbar(rows['Learning rate'], rows['Reduction @ 8K (%)'], yerr=rows['Seed noise @ 8K (%)'],
                      marker='o', capsize=3, color=schedule_colors[schedule], label=schedule)
    axis.axhline(100 * adoption['rule']['minimum_mean_reduction'], color='black', linestyle='--', linewidth=1,
                 label='adoption threshold')
    axis.axhline(0, color='grey', linewidth=0.8)
    axis.set(xscale='log', xlabel='Learning rate', title=f'Training pairs from {context:,}-token context',
             ylabel='Mean 8K KL reduction vs current (%)' if context == 128 else '')
axes[0].legend(frameon=False)
figure.suptitle('Error bars: the rule\'s seed noise (the current setting shows its own seed spread)', y=1.02)
figure.tight_layout()
plt.show()

context_rows = []
for (schedule, rate), rows in grid_df.groupby(['Schedule', 'Learning rate']):
    by_context = rows.set_index('Training context')
    if not {128, 8192} <= set(by_context.index):
        continue
    context_rows.append({
        'Schedule': schedule,
        'Learning rate': rate,
        '8K score, 128 training': by_context.loc[128, 'Reduction @ 8K (%)'],
        '8K score, 8K training': by_context.loc[8192, 'Reduction @ 8K (%)'],
        '128 score, 128 training': by_context.loc[128, 'Reduction @ 128 (%)'],
        '128 score, 8K training': by_context.loc[8192, 'Reduction @ 128 (%)'],
    })
display(
    pd.DataFrame(context_rows).style.hide(axis='index')
    .format({'Learning rate': '{:g}', **{column: '{:+.2f}' for column in (
        '8K score, 128 training', '8K score, 8K training', '128 score, 128 training', '128 score, 8K training',
    )}})
    .set_caption('Mean KL reduction vs the current setting (%) by training context; positive is better')
)

##### Gradient training versus exact least squares

In [ ]:
current_fits = fits_df[fits_df['Setting'] == CURRENT_SETTING]
solve_rows = [{
    'Setting': CURRENT_SETTING, 'Reduction @ 8K': 0.0, 'Reduction @ 128': 0.0,
    'Mean NMSE @ 128': current_fits['NMSE @ 128'].mean(), 'Largest Gram condition number': None,
    'Mean fit seconds': current_fits['Fit seconds'].mean(),
}]
for name in ('ols-128', 'ols-8k'):
    fits = [fit for fit in block_results['fits'] if fit['setting'] == name]
    solve_rows.append({
        'Setting': name,
        'Reduction @ 8K': adoption['settings'][name]['mean_reduction']['8192'],
        'Reduction @ 128': adoption['settings'][name]['mean_reduction']['128'],
        'Mean NMSE @ 128': np.mean([fit['validation_128']['nmse'] for fit in fits]),
        'Largest Gram condition number': max(fit['least_squares']['gram_condition_number'] for fit in fits),
        'Mean fit seconds': np.mean([fit['fit_seconds'] for fit in fits]),
    })
display(
    pd.DataFrame(solve_rows).style.hide(axis='index')
    .format({'Reduction @ 8K': '{:+.2%}', 'Reduction @ 128': '{:+.2%}', 'Mean NMSE @ 128': '{:.4f}',
             'Largest Gram condition number': '{:.3g}', 'Mean fit seconds': '{:.0f}'}, na_rep='-')
    .set_caption(
        'Least squares keeps the teacher-slice gate and up projections and solves the down projection exactly; '
        'it minimizes training error, not single-block KL'
    )
)

##### Batch size

In [ ]:
batch_rows = []
for name in (CURRENT_SETTING, f'{CURRENT_SETTING}-b256'):
    rows = fits_df[fits_df['Setting'] == name]
    batch_rows.append({
        'Setting': name,
        'Batch': setting_grid[name]['batch_size'],
        'Reduction @ 8K': 0.0 if name == CURRENT_SETTING else adoption['settings'][name]['mean_reduction']['8192'],
        'Seed noise @ 8K': reference_noise if name == CURRENT_SETTING else adoption['settings'][name]['mean_seed_noise'],
        'Mean best epoch': rows['Best epoch'].mean(),
        'Mean updates': rows['Updates'].mean(),
        'Mean fit seconds': rows['Fit seconds'].mean(),
    })
display(
    pd.DataFrame(batch_rows).style.hide(axis='index')
    .format({'Reduction @ 8K': '{:+.2%}', 'Seed noise @ 8K': '{:.2%}', 'Mean best epoch': '{:.1f}',
             'Mean updates': '{:,.0f}', 'Mean fit seconds': '{:.0f}'})
    .set_caption('Batch 256 against batch 2,048 under the conditions of this study')
)
if 'operator-distillation' in earlier_evidence:
    earlier_batch = pd.DataFrame([
        {'Calibration pairs': row['calibration_tokens'], 'NMSE, batch 2,048': row['original_nmse'],
         'NMSE, batch 256': row['optimized_nmse']}
        for row in earlier_evidence['operator-distillation']['results']['combined']['comparison']
    ])
    display(
        earlier_batch.style.hide(axis='index').format({'Calibration pairs': '{:,}', 'NMSE, batch 2,048': '{:.4f}',
                                                       'NMSE, batch 256': '{:.4f}'})
        .set_caption('Earlier evidence (operator distillation; layer 11, random initialization, one seed): '
                     'trained to early stopping, both batch sizes reach the same fit')
    )
else:
    print('Operator distillation, section 2: at 98,304 pairs batch 2,048 reaches NMSE 0.4666 and batch 256 0.4668.')

##### Calibration size

In [ ]:
if 'swiglu-3' in earlier_evidence:
    calibration = earlier_evidence['swiglu-3']['results']['calibration']
    calibration_df = pd.DataFrame([
        {
            'Calibration pairs per block': row['calibration_pairs'],
            'Allocation-selection KL': row['allocation_selection']['teacher_kl'],
            'WikiText validation PPL (pre-recovery)': row['wikitext_validation']['perplexity'],
        }
        for row in calibration['model_evaluation']
    ])
    display(
        calibration_df.style.hide(axis='index')
        .format({'Calibration pairs per block': '{:,}', 'Allocation-selection KL': '{:.6f}',
                 'WikiText validation PPL (pre-recovery)': '{:.4f}'})
        .set_caption(f"SwiGLU-3, 50% allocation: {calibration['selected_calibration_pairs']:,} pairs selected "
                     f"({calibration['selection_rule']})")
    )
else:
    print('SwiGLU-3 (swiglu-progression.md): allocation-selection KL 0.4025, 0.3889, and 0.3739 at 98,304, '
          '196,608, and 393,216 pairs; 393,216 was selected.')
print('Part A keeps 393,216 pairs. Stage 2 tests 786,432 pairs only if the stage-2 indicator reports overfitting.')

##### Initialization

In [ ]:
if 'operator-distillation' in earlier_evidence:
    initialization_df = pd.DataFrame([
        {
            'Initialization': row['initialization'],
            'Initial NMSE': row['initial_nmse'],
            'Fitted NMSE': row['fitted_nmse'],
            'Best epoch': row['best_epoch'],
            'Updates': row['completed_optimizer_updates'],
        }
        for row in earlier_evidence['operator-distillation']['results']['teacher_initialization']['summary']
    ])
    display(
        initialization_df.style.hide(axis='index')
        .format({'Initial NMSE': '{:.4f}', 'Fitted NMSE': '{:.4f}'})
        .set_caption('Earlier evidence (operator distillation; layer 11, 50% width, 98,304 pairs, one seed)')
    )
else:
    print('Operator distillation, section 3: fitted NMSE 0.4666 from random weights, 0.2655 from a random teacher '
          'subset, and 0.2038 from the importance-ranked teacher subset.')
slice_nmse = fits_df[fits_df['Setting'] == 'slice']['NMSE @ 128'].mean()
print(f"Part A: the unfitted importance slice has mean NMSE @ 128 {slice_nmse:.4f}; "
      f"{CURRENT_SETTING} reaches {current_fits['NMSE @ 128'].mean():.4f}.")

##### Training dynamics

In [ ]:
gradient_settings = [name for name, setting in setting_grid.items() if setting['method'] == 'gradient']
dynamics_df = (
    fits_df[fits_df['Setting'].isin(gradient_settings)]
    .groupby(['Setting', 'Width ratio'], observed=True)
    .agg(**{'Mean best epoch': ('Best epoch', 'mean'), 'Max best epoch': ('Best epoch', 'max'),
            'Mean epochs': ('Epochs', 'mean')})
    .reset_index()
    .pivot(index='Setting', columns='Width ratio', values=['Mean best epoch', 'Max best epoch', 'Mean epochs'])
)
display(dynamics_df.style.format('{:.1f}').set_caption(
    'When fits stop improving; cosine settings always run 32 epochs and keep their best epoch'
))
print(f"Stage-2 indicator ({stage_two.get('definition', 'not applicable')}): "
      f"overfitting layers {stage_two.get('overfitting_layers', [])}; stage 2 needs the researcher's go.")

##### Conclusion and limits

In [ ]:
selected = adoption['selected_recipe']
print(f"Part B fits with {selected}: {describe(selected)}")
if selected == CURRENT_SETTING:
    closest = max(adoption['settings'], key=lambda name: adoption['settings'][name]['mean_reduction']['8192'])
    row = adoption['settings'][closest]
    print(f"No tested setting cleared the rule; the closest was {closest} at "
          f"{row['mean_reduction']['8192']:+.2%} ({'; '.join(row['reasons'])}).")
else:
    row = adoption['settings'][selected]
    print(f"It reduces mean 8K single-block KL by {row['mean_reduction']['8192']:.2%} against a seed noise of "
          f"{row['mean_seed_noise']:.2%}; worst combination {row['worst_reduction']:+.2%}.")

Limits of this evidence:

- Three layers (3, 11, and 19) at two widths (25% and 50%). Other layers and the narrow widths 512 and 1,024 are not tested.
- Seeds vary only the batch order; a different calibration sample could move the results more.
- One model (SmolLM2-1.7B) and one local objective (activation MSE).
- The 128-token score uses 3,072 tokens. Every setting is scored on the same windows, so differences between settings are less noisy than the absolute values.
- The batch-size, calibration-size, and initialization evidence from earlier studies is narrower and is labelled as such.

#### Part B: preparation

In [ ]:
if not PREPARED_PATH.is_file():
    raise FileNotFoundError(f'Part B has not been prepared yet: {PREPARED_PATH}')

prepared = json.loads(PREPARED_PATH.read_text(encoding='utf-8'))
if (prepared.get('workflow'), prepared.get('command')) != ('swiglu-8', 'prepare'):
    raise ValueError('Unexpected SwiGLU-8 preparation artifact')

prepared_results = prepared['results']
print(f"Preparation: {PREPARED_PATH.relative_to(PROJECT_ROOT)}")
print(f"Status: {prepared['status']}" + (f"; error: {prepared['error']}" if prepared.get('error') else ''))
print(f"Fitting recipe: {prepared['configuration']['part_b']['fitting_recipe']}")
print(f"Recovery stream: {prepared_results['stream']['token_count']:,} tokens, sha256 {prepared_results['stream']['sha256']}")

##### Acceptance

In [ ]:
preparation_acceptance = acceptance_frame(prepared_results.get('acceptance', {}))
display(
    acceptance_summary(preparation_acceptance).style.hide(axis='index')
    .format({'Largest relative difference': '{:.2e}'}, na_rep='-')
    .set_caption('P1 selection audit, P2 start fits, P3 dense reference, P4 recovery stream, P5 8K KL')
)
shown = preparation_acceptance[
    preparation_acceptance['Group'].isin(['P3', 'P4', 'P5']) | ~preparation_acceptance['Passed'].astype(bool)
]
display(shown.style.hide(axis='index').format(ACCEPTANCE_FORMAT, na_rep='-')
        .set_caption('P3-P5 checks and any failed P1/P2 check'))

##### Allocation

In [ ]:
allocation_summary = []
allocation_rows = []
for target_text, entry in prepared_results['allocation']['targets'].items():
    target = float(target_text)
    solver = entry['solver']
    allocation_summary.append({
        'Target': target,
        'Width source': solver['source'],
        'Realized MLP removal': solver['realized_mlp_removal'],
        'Predicted teacher KL': solver['predicted_teacher_kl'],
        'Boundary layer': solver['boundary_layer'],
        'Replacement layers': sum(not row['retains_dense_module'] for row in entry['allocation']),
    })
    for row in entry['allocation']:
        allocation_rows.append({
            'Target': target,
            'Layer': row['layer'],
            'Width ratio': row['replacement_width'] / row['original_width'],
        })
allocation_df = pd.DataFrame(allocation_rows)
display(
    pd.DataFrame(allocation_summary).style.hide(axis='index')
    .format({'Target': '{:.0%}', 'Realized MLP removal': '{:.4%}', 'Predicted teacher KL': '{:.5f}'}, na_rep='-')
    .set_caption('Shared allocations; both arms use these widths and neurons')
)
width_matrix = allocation_df.pivot(index='Target', columns='Layer', values='Width ratio').sort_index()
figure, axis = plt.subplots(figsize=(14, 4.4))
image = axis.imshow(width_matrix, aspect='auto', cmap='viridis', vmin=0.0625, vmax=1.0)
axis.set(
    xlabel='Transformer layer',
    ylabel='Eligible-MLP removal target',
    xticks=np.arange(width_matrix.shape[1]),
    yticks=np.arange(width_matrix.shape[0]),
    xticklabels=width_matrix.columns,
    yticklabels=[f'{target:.0%}' for target in width_matrix.index],
    title='Retained-width profiles (20-50% copied from SwiGLU-7)',
)
axis.grid(False)
colorbar = figure.colorbar(image, ax=axis, pad=0.02)
colorbar.set_label('Retained width / original width')
figure.tight_layout()
plt.show()

In [ ]:
resolve_df = pd.DataFrame([
    {
        'Target': row['target'],
        'Changed layers': len(row['changed_layers']),
        'Changes': ', '.join(
            f"L{change['layer']}: {change['swiglu7_width']} -> {change['resolved_width']}"
            for change in row['changed_layers']
        ) or '-',
        'SwiGLU-7 predicted KL': row['swiglu7_predicted_teacher_kl'],
        'Re-solved predicted KL': row['resolved_predicted_teacher_kl'],
    }
    for row in prepared_results['allocation']['swiglu7_width_resolve_report']
])
display(
    resolve_df.style.hide(axis='index')
    .format({'Target': '{:.0%}', 'SwiGLU-7 predicted KL': '{:.5f}', 'Re-solved predicted KL': '{:.5f}'})
    .set_caption('Would the extended width menu change the 20-50% allocations? Reported only; the runs keep SwiGLU-7 widths')
)

curve_points = pd.DataFrame(prepared_results['width_curves']['rows'])
curve_points = curve_points[curve_points['replacement_width'] < ORIGINAL_WIDTH]
figure, axis = plt.subplots(figsize=(9, 4.8))
layer_colors = dict(zip(sorted(curve_points['layer'].unique()), sns.color_palette('viridis', curve_points['layer'].nunique())))
for layer, rows in curve_points.groupby('layer'):
    rows = rows.sort_values('replacement_width')
    axis.plot(rows['replacement_width'] / ORIGINAL_WIDTH, rows['monotone_teacher_kl'],
              color=layer_colors[layer], linewidth=1.2, marker='o', markersize=3)
axis.set(
    yscale='log', xlabel='Retained width / original width', ylabel='Single-block KL @ 128 (monotone)',
    title=f"Width curves ({prepared_results['width_curves']['recipe_id']}); 0.0625 and 0.125 are new, one line per layer",
)
figure.tight_layout()
plt.show()

#### Part B: recovery

In [ ]:
runs = {}
for (arm, target), path in RUN_PATHS.items():
    if not path.is_file():
        continue
    run = json.loads(path.read_text(encoding='utf-8'))
    if (run.get('workflow'), run.get('command')) != ('swiglu-8', 'train') or run.get('identity') != {'arm': arm, 'target': target}:
        raise ValueError(f'Artifact identity mismatch: {path}')
    runs[(arm, target)] = run
completed_runs = {key: run for key, run in runs.items() if run['status'] == 'completed'}
missing = [f'{arm} @ {target:.0%}' for arm, target in RUN_PATHS if (arm, target) not in runs]
unfinished = [f"{arm} @ {target:.0%} ({run['status']})" for (arm, target), run in runs.items() if run['status'] != 'completed']
print(f'Run artifacts: {len(runs)} of {len(RUN_PATHS)}; completed: {len(completed_runs)}')
print(f"Missing: {', '.join(missing) or 'none'}")
print(f"Not completed: {', '.join(unfinished) or 'none'}")

In [ ]:
recovery_df = pd.DataFrame([
    {
        'Arm': arm,
        'Target': target,
        'Requested tokens': max(row['requested_tokens']),
        'Tokens': row['actual_tokens'],
        'Validation KL': row['recovery_validation_kl'],
        'Monitoring PPL': row.get('wikitext_validation', {}).get('perplexity'),
    }
    for (arm, target), run in runs.items()
    for row in run['results'].get('recovery', {}).get('validation_history', [])
], columns=['Arm', 'Target', 'Requested tokens', 'Tokens', 'Validation KL', 'Monitoring PPL'])
recovery_df = recovery_df.sort_values(['Target', 'Arm', 'Tokens'])
figure, axes = plt.subplots(2, 4, figsize=(17, 7.5), sharex=True)
for axis, target in zip(axes.flat, TARGETS):
    for arm in ARMS:
        rows = recovery_df[(recovery_df['Target'] == target) & (recovery_df['Arm'] == arm)]
        axis.plot(rows['Tokens'] / 1e6, rows['Validation KL'], marker='o', markersize=3,
                  color=ARM_COLORS[arm], label=ARM_LABELS[arm])
    axis.set(title=f'{target:.0%} eligible-MLP removal', xlabel='Recovery tokens (M)', ylabel='Monitoring KL')
axes[0, 0].legend(frameon=False)
figure.suptitle('Recovery trajectories: slice only versus fitted', y=1.01)
figure.tight_layout()
plt.show()

In [ ]:
paired = recovery_df.pivot_table(index=['Target', 'Requested tokens'], columns='Arm', values='Validation KL')
if not set(ARMS) <= set(paired.columns):
    gap_df = pd.DataFrame(columns=['Target', 'Requested tokens', 'Monitoring KL gap'])
    print('No target has monitoring rows for both arms yet.')
else:
    gap_df = paired.dropna().reset_index()
    gap_df['Monitoring KL gap'] = (gap_df['slice'] - gap_df['fitted']) / gap_df['fitted']
    display(
        gap_df.pivot(index='Target', columns='Requested tokens', values='Monitoring KL gap')
        .style.format('{:+.2%}', na_rep='-')
        .set_caption('Monitoring-KL gap (slice - fitted) / fitted by requested recovery tokens')
    )
    figure, axis = plt.subplots(figsize=(9, 4.8))
    target_colors = dict(zip(TARGETS, sns.color_palette('viridis', len(TARGETS))))
    for target, rows in gap_df.groupby('Target'):
        axis.plot(rows['Requested tokens'] / 1e6, 100 * rows['Monitoring KL gap'], marker='o',
                  color=target_colors[target], label=f'{target:.0%}')
    axis.axhspan(-100 * GAP_THRESHOLD, 100 * GAP_THRESHOLD, color='grey', alpha=0.15, linewidth=0)
    axis.set(xlabel='Recovery tokens (M)', ylabel='Gap (%)', title='Does recovery close the local-fitting gap?')
    axis.legend(frameon=False, title='Target', ncol=2)
    figure.tight_layout()
    plt.show()

##### Reproduction of SwiGLU-7

In [ ]:
reproduction_frames = [
    acceptance_frame(run['results'].get('acceptance', {})).assign(Run=f'fitted @ {target:.0%}')
    for (arm, target), run in runs.items() if run.get('reproduction_check')
]
if not reproduction_frames:
    print('No reproduction runs; B1-B2 apply only to the fitted 20-50% runs while the current recipe is kept.')
else:
    reproduction_df = pd.concat(reproduction_frames, ignore_index=True)
    display(
        reproduction_df.groupby('Run').agg(
            Checks=('Passed', 'size'), Passed=('Passed', 'sum'),
            **{'Largest relative difference': ('Relative difference', 'max')},
        ).reset_index().style.hide(axis='index').format({'Largest relative difference': '{:.2e}'})
        .set_caption('B1-B2: fitted runs against the first 100M tokens of SwiGLU-7 S7-0')
    )
    display(
        reproduction_df[['Run', 'Check', 'Metric', 'Tokens', 'Value', 'Reference', 'Relative difference', 'Tolerance', 'Passed']]
        .style.hide(axis='index').format(ACCEPTANCE_FORMAT, na_rep='-')
    )

#### Part B: final evaluation

In [ ]:
dense = prepared_results['dense_evaluation']
final_df = pd.DataFrame([
    {
        'Arm': arm,
        'Target': target,
        'Realized MLP removal': prepared_results['starts'][f'{arm}:{target}']['realized_eligible_mlp_removal'],
        'Whole-model removal': run['results']['evaluation']['footprint']['whole_model_parameter_removal'],
        'KL @ 8K': run['results']['evaluation']['teacher_kl_8192']['kl'],
        'Monitoring KL @ 100M': run['results']['recovery']['validation_history'][-1]['recovery_validation_kl'],
        'Test PPL @ 128': run['results']['evaluation']['likelihood']['test-128']['perplexity'],
        'Test PPL @ 2,048': run['results']['evaluation']['likelihood']['test-2048']['perplexity'],
        'Test PPL @ 8,192': run['results']['evaluation']['likelihood']['test-8192']['perplexity'],
        'Macro accuracy': run['results']['evaluation']['comparison_to_dense']['macro_accuracy'],
        'Macro delta vs dense': run['results']['evaluation']['comparison_to_dense']['macro_delta'],
        'Bundle GiB': run['results']['evaluation']['footprint']['bundle_bytes'] / 2**30,
    }
    for (arm, target), run in completed_runs.items()
], columns=[
    'Arm', 'Target', 'Realized MLP removal', 'Whole-model removal', 'KL @ 8K', 'Monitoring KL @ 100M',
    'Test PPL @ 128', 'Test PPL @ 2,048', 'Test PPL @ 8,192', 'Macro accuracy', 'Macro delta vs dense', 'Bundle GiB',
]).sort_values(['Target', 'Arm'])
display(
    final_df.style.hide(axis='index')
    .format({
        'Target': '{:.0%}', 'Realized MLP removal': '{:.2%}', 'Whole-model removal': '{:.2%}',
        'KL @ 8K': '{:.5f}', 'Monitoring KL @ 100M': '{:.5f}', 'Test PPL @ 128': '{:.3f}',
        'Test PPL @ 2,048': '{:.3f}', 'Test PPL @ 8,192': '{:.3f}', 'Macro accuracy': '{:.2%}',
        'Macro delta vs dense': '{:+.2%}', 'Bundle GiB': '{:.2f}',
    })
    .set_caption('Final evaluation at 100M tokens (8K KL on the reloaded BF16 bundle)')
)
print(f"Dense test PPL @ 8,192: {dense['likelihood']['test-8192']['perplexity']:.4f}")

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for arm in ARMS:
    rows = final_df[final_df['Arm'] == arm].sort_values('Realized MLP removal')
    for axis, metric, scale in (
        (axes[0], 'KL @ 8K', 1), (axes[1], 'Test PPL @ 8,192', 1), (axes[2], 'Macro accuracy', 100),
    ):
        axis.plot(100 * rows['Realized MLP removal'], scale * rows[metric], marker='o',
                  color=ARM_COLORS[arm], label=ARM_LABELS[arm])
axes[1].axhline(dense['likelihood']['test-8192']['perplexity'], color='black', linestyle=':', linewidth=1, label='Dense')
axes[0].set(xlabel='Eligible-MLP removal (%)', ylabel='Teacher KL @ 8K', title='8K teacher KL')
axes[1].set(xlabel='Eligible-MLP removal (%)', ylabel='Test PPL @ 8,192', title='Likelihood')
axes[2].set(xlabel='Eligible-MLP removal (%)', ylabel='Macro accuracy (%)', title='Zero-shot tasks')
axes[1].legend(frameon=False)
figure.suptitle('Homogeneous SwiGLU control spectrum at 100M recovery tokens', y=1.02)
figure.tight_layout()
plt.show()

task_df = pd.DataFrame([
    {
        'Arm': arm,
        'Target': target,
        'Task': task,
        'Accuracy': row['student'],
        'Paired delta vs dense': row['student_minus_dense'],
        'CI95 low': row['ci95'][0],
        'CI95 high': row['ci95'][1],
    }
    for (arm, target), run in completed_runs.items()
    for task, row in run['results']['evaluation']['comparison_to_dense']['tasks'].items()
], columns=['Arm', 'Target', 'Task', 'Accuracy', 'Paired delta vs dense', 'CI95 low', 'CI95 high'])
display(
    task_df.sort_values(['Target', 'Task', 'Arm']).style.hide(axis='index')
    .format({'Target': '{:.0%}', 'Accuracy': '{:.2%}', 'Paired delta vs dense': '{:+.2%}',
             'CI95 low': '{:+.2%}', 'CI95 high': '{:+.2%}'})
    .set_caption('Per-task accuracy with paired bootstrap intervals against the dense model')
)

#### Interpretation

In [ ]:
def gap_band(value):
    if abs(value) <= GAP_THRESHOLD:
        return 'within threshold'
    return 'fitted better' if value > 0 else 'slice better'


interpretation_rows = []
for target in TARGETS:
    if not all((arm, target) in completed_runs for arm in ARMS):
        continue
    kl = {arm: completed_runs[(arm, target)]['results']['evaluation']['teacher_kl_8192']['kl'] for arm in ARMS}
    ppl = {
        arm: completed_runs[(arm, target)]['results']['evaluation']['likelihood']['test-8192']['perplexity']
        for arm in ARMS
    }
    macro = {
        arm: completed_runs[(arm, target)]['results']['evaluation']['comparison_to_dense']['macro_accuracy']
        for arm in ARMS
    }
    monitoring = gap_df[gap_df['Target'] == target].set_index('Requested tokens')['Monitoring KL gap']
    gap = (kl['slice'] - kl['fitted']) / kl['fitted']
    change = monitoring.loc[100_000_000] - monitoring.loc[50_000_000]
    stable = abs(change) < STABILITY_THRESHOLD
    if abs(gap) <= GAP_THRESHOLD:
        conclusion = 'Fitting does not matter after recovery'
    elif stable and gap > 0:
        conclusion = 'Fitting matters'
    elif stable:
        conclusion = 'Fitting hurts (slice only is better)'
    else:
        conclusion = 'Inconclusive at 100M'
    ppl_gap = (ppl['slice'] - ppl['fitted']) / ppl['fitted']
    interpretation_rows.append({
        'Target': target,
        '8K KL gap': gap,
        'Monitoring gap @ 0': monitoring.loc[0],
        'Monitoring gap @ 50M': monitoring.loc[50_000_000],
        'Monitoring gap @ 100M': monitoring.loc[100_000_000],
        'Change 50M->100M (pp)': 100 * change,
        'Stable': stable,
        'Conclusion': conclusion,
        'Test PPL @ 8,192 gap': ppl_gap,
        'PPL agrees with KL': gap_band(ppl_gap) == gap_band(gap),
        'Macro accuracy fitted - slice (pp)': 100 * (macro['fitted'] - macro['slice']),
    })
interpretation_df = pd.DataFrame(interpretation_rows)
if interpretation_df.empty:
    print('No target has completed runs for both arms yet.')
else:
    display(
        interpretation_df.style.hide(axis='index')
        .format({
            'Target': '{:.0%}', '8K KL gap': '{:+.2%}', 'Monitoring gap @ 0': '{:+.2%}',
            'Monitoring gap @ 50M': '{:+.2%}', 'Monitoring gap @ 100M': '{:+.2%}',
            'Change 50M->100M (pp)': '{:+.2f}', 'Test PPL @ 8,192 gap': '{:+.2%}',
            'Macro accuracy fitted - slice (pp)': '{:+.2f}',
        })
        .set_caption(
            f'Gap = (slice - fitted) / fitted; within {GAP_THRESHOLD:.0%} means fitting does not matter '
            f'(a judgment call); stable means a change below {100 * STABILITY_THRESHOLD:.0f} pp from 50M to 100M'
        )
    )
    display(interpretation_df['Conclusion'].value_counts().rename('Targets').to_frame())

#### Compute and deployment

In [ ]:
cost_df = pd.DataFrame([
    {
        'Arm': arm,
        'Target': target,
        'Status': run['status'],
        'Training hours': run['results'].get('recovery', {}).get('training_seconds', 0.0) / 3600,
        'Monitoring hours': run['results'].get('recovery', {}).get('evaluation_seconds', 0.0) / 3600,
        'Checkpoint hours': run['results'].get('recovery', {}).get('checkpoint_seconds', 0.0) / 3600,
        'Final evaluation hours': run['results'].get('final_evaluation_wall_seconds', 0.0) / 3600,
        'Workflow span hours': workflow_hours(run),
        'Peak RAM (GiB)': run['results'].get('recovery', {}).get('memory', {}).get('peak_ram_gib'),
        'Peak VRAM (GiB)': run['results'].get('recovery', {}).get('memory', {}).get('peak_vram_gib'),
        'Trainable parameters': run['results'].get('trainable_scope', {}).get('trainable_parameters'),
    }
    for (arm, target), run in runs.items()
], columns=[
    'Arm', 'Target', 'Status', 'Training hours', 'Monitoring hours', 'Checkpoint hours',
    'Final evaluation hours', 'Workflow span hours', 'Peak RAM (GiB)', 'Peak VRAM (GiB)', 'Trainable parameters',
]).sort_values(['Target', 'Arm'])
display(
    cost_df.style.hide(axis='index')
    .format({
        'Target': '{:.0%}', 'Training hours': '{:.2f}', 'Monitoring hours': '{:.2f}', 'Checkpoint hours': '{:.2f}',
        'Final evaluation hours': '{:.2f}', 'Workflow span hours': '{:.2f}', 'Peak RAM (GiB)': '{:.2f}',
        'Peak VRAM (GiB)': '{:.2f}', 'Trainable parameters': '{:,.0f}',
    }, na_rep='-')
    .set_caption('Per-run workflow timing and memory on one H200')
)
# The block check ran as parallel tasks; its GPU time is the sum of the task spans.
block_tasks = block_results.get('tasks') or [block_check]
stage_df = pd.DataFrame([
    {'Stage': 'block-check', 'Jobs': len(block_tasks), 'Planned GPU-h': PLANNED_GPU_HOURS['block-check'],
     'Workflow span GPU-h': sum(workflow_hours(task) for task in block_tasks)},
    {'Stage': 'prepare', 'Jobs': 1, 'Planned GPU-h': PLANNED_GPU_HOURS['prepare'],
     'Workflow span GPU-h': workflow_hours(prepared)},
    {'Stage': 'train', 'Jobs': len(runs), 'Planned GPU-h': PLANNED_GPU_HOURS['train'],
     'Workflow span GPU-h': cost_df['Workflow span hours'].sum()},
])
stage_df['Within plan'] = [measured <= high for (low, high), measured in zip(stage_df['Planned GPU-h'], stage_df['Workflow span GPU-h'])]
stage_df['Planned GPU-h'] = [f'{low:g}-{high:g}' for low, high in stage_df['Planned GPU-h']]
display(stage_df.style.hide(axis='index').format({'Workflow span GPU-h': '{:.2f}'})
        .set_caption('Planned versus measured; spans are workflow proxies, not scheduler allocation'))
print(f"Total workflow span: {stage_df['Workflow span GPU-h'].sum():.2f} GPU-hours (planned about 39-63)")
print(f"Final bundles on PROJECT: {final_df['Bundle GiB'].sum():.1f} GiB across {len(final_df)} completed runs")
print('Scheduler allocation, failures, and retries are recorded in docs/infrastructure/perun/perun-log.md. '
      'A resumed job span includes the time between attempts.')

In [ ]:
environment_df = pd.DataFrame([
    {
        'Artifact': name,
        'Python': artifact['environment']['python'],
        'PyTorch': artifact['environment']['packages']['torch'],
        'Transformers': artifact['environment']['packages']['transformers'],
        'lm-eval': artifact['environment']['packages']['lm_eval'],
        'CUDA runtime': artifact['environment']['cuda_runtime'],
        'GPU': artifact['environment']['gpu'],
    }
    for name, artifact in (
        ('block-check', block_check),
        ('prepare', prepared),
        *((f'{arm} @ {target:.0%}', run) for (arm, target), run in runs.items()),
    )
])
display(environment_df.drop_duplicates(subset=environment_df.columns[1:].tolist())
        .style.hide(axis='index').set_caption('Distinct software and hardware environments'))